In [38]:
%run PAClient.ipynb

In [2]:
import unittest
from datetime import datetime

In [3]:
URL="https://5ftnewdis2.execute-api.us-east-1.amazonaws.com/test"

In [4]:
class TestPA(unittest.TestCase):
    
    def setUp(self):
        self.client = PAClient(url=URL)
        
    ### USERS
    
    #@unittest.skip
    def test_create_user(self):
        response = self.client.userCreate("Test User", 445)
        self.assertEqual(response["name"], "Test User")
        self.assertEqual(response["balance"], 445)
        self.assertIsInstance(response["uid"], str)
        current_date = datetime.now().date().isoformat()
        created_date = datetime.strptime(response['datecreated'], '%Y-%m-%d %H:%M:%S.%f').date().isoformat()
        self.assertEqual(created_date, current_date)
        self.client.userDelete(response["uid"])
    
    #@unittest.skip
    def test_get_user(self):
        user_create_response = self.client.userCreate("Test User", 100)
        response = self.client.userGet(user_create_response['uid'])
        self.assertEqual(response["name"], "Test User")
        self.assertEqual(response["balance"], 100)
        self.assertEqual(response["uid"], user_create_response['uid'])
        current_date = datetime.now().date().isoformat()
        created_date = datetime.strptime(response['datecreated'], '%Y-%m-%d %H:%M:%S.%f').date().isoformat()
        self.assertEqual(created_date, current_date)
        self.client.userDelete(response["uid"])

    #@unittest.skip
    def test_credit_user(self):
        user_create_response = self.client.userCreate("Test User", 100)
        user_credit_response = self.client.userCredit(user_create_response["uid"], 50)
        user_credit_response = self.client.userCredit(user_credit_response['uid'], -80.5)
        user_get_response = self.client.userGet(user_credit_response["uid"])
        self.assertEqual(user_get_response["balance"], 100+50-80.5)
        self.client.userDelete(user_create_response["uid"])

    #@unittest.skip
    def test_delete_user(self):
        user_create_response = self.client.userCreate("Test User", 100)
        response = self.client.userDelete(user_create_response["uid"])
        self.assertEqual(response, user_create_response["uid"])
        with self.assertRaises(json.JSONDecodeError):
            self.client.userGet(user_create_response["uid"])
    
    #@unittest.skip
    def test_enum_user(self):
        user_create_response = self.client.userCreate("Test User", 100)
        user_enum_response = self.client.userEnum(q=f"uid='{user_create_response['uid']}'")
        self.assertEqual(user_enum_response[0]["uid"], user_create_response['uid'])
        self.client.userDelete(user_create_response["uid"])
        
    ### DOCS   
    #@unittest.skip
    def test_create_doc(self):
        user_create_response = self.client.userCreate("Test User", 100)
        response = self.client.docCreate(content={"ssText":"test doc"},owner=user_create_response['uid'])
        self.assertEqual(response["content"], {"ssText":"test doc"})
        self.assertEqual(response["status"], "ready")
        self.assertEqual(response["owner"], user_create_response['uid'])
        self.assertIsInstance(response["uid"], str)
        current_date = datetime.now().date().isoformat()
        created_date = datetime.strptime(response['datecreated'], '%Y-%m-%d %H:%M:%S.%f').date().isoformat()
        self.assertEqual(created_date, current_date)
        self.client.docDelete(response["uid"])
        self.client.userDelete(user_create_response["uid"])

    #@unittest.skip
    def test_update_doc(self):
        user_create_response = self.client.userCreate("Test User", 100)
        doc_create_response = self.client.docCreate(content={"ssText":"test doc"},owner=user_create_response['uid'])
        doc_update_response = self.client.docUpdate(doc_create_response["uid"],content={"ssText":"test doc 2"}, status="completed")
        doc_format_response = self.client.docFormat(doc_create_response["uid"])
        self.assertEqual(doc_format_response["content"], {"ssText":"test doc 2"})
        self.assertEqual(doc_format_response["status"], "completed")
        self.assertEqual(doc_format_response["owner"], user_create_response['uid'])
        self.assertEqual(doc_format_response['datecreated'], doc_create_response['datecreated'])
        self.client.docDelete(doc_format_response["uid"])
        self.client.userDelete(user_create_response["uid"])
    
    #@unittest.skip
    def test_enum_doc(self):
        user_create_response = self.client.userCreate("Test User", 100)
        doc_create_response = self.client.docCreate(content={"ssText":"test doc"},owner=user_create_response['uid'])
        doc_enum_response = self.client.docEnum(q=f"uid='{doc_create_response['uid']}'")
        self.assertEqual(doc_enum_response[0]["uid"], doc_create_response['uid'])
        self.client.docDelete(doc_create_response["uid"])
        self.client.userDelete(user_create_response["uid"])
    
    #@unittest.skip
    def test_doc_media_image(self):
        user_create_response = self.client.userCreate("Test User", 100)
        test_image_path="image2text.jpg"
        doc_create_response = self.client.docCreate(content={"bjImage":file2b64(test_image_path)},owner=user_create_response['uid'])
        doc_format_response = self.client.docFormat(doc_create_response['uid'],fmt="json_base64")
        self.assertEqual(doc_format_response["content"]["bjImage"], file2b64(test_image_path))
        self.client.docDelete(doc_create_response["uid"])
        self.client.userDelete(user_create_response["uid"])
    
    #@unittest.skip
    def test_doc_media_audio(self):
        user_create_response = self.client.userCreate("Test User", 100)
        test_audio_path="speech2Text.mp3"
        doc_create_response = self.client.docCreate(content={"bmAudio":file2b64(test_audio_path)},owner=user_create_response['uid'])
        doc_format_response = self.client.docFormat(doc_create_response['uid'],fmt="json_base64")
        self.assertEqual(doc_format_response["content"]["bmAudio"], file2b64(test_audio_path))
        self.client.docDelete(doc_create_response["uid"])
        self.client.userDelete(user_create_response["uid"])
    
    #@unittest.skip
    def test_delete_doc(self):
        user_create_response = self.client.userCreate("Test User", 100)
        doc_create_response = self.client.docCreate(content={"ssText":"test doc"},owner=user_create_response['uid'])
        response = self.client.docDelete(doc_create_response["uid"])
        self.assertEqual(response, doc_create_response["uid"])
        with self.assertRaises(json.JSONDecodeError):
            self.client.docFormat(doc_create_response["uid"])
        self.client.userDelete(user_create_response["uid"])
        
    ### TRANSFORMERS
    #@unittest.skip
    def test_create_transformer(self):
        user_create_response = self.client.userCreate("Test User", 100)
        response = self.client.transformCreate("Test Transformer", {"chain":"_dbg_text2text"},owner=user_create_response['uid'])
        self.assertEqual(response["cfg"], {"chain":"_dbg_text2text"})
        self.assertEqual(response["name"], "Test Transformer")
        self.assertEqual(response["owner"], user_create_response['uid'])
        self.assertIsInstance(response["uid"], str)
        self.assertEqual(response['fees'], {'process_fees': {}, 'prompt_fees': {}})
        current_date = datetime.now().date().isoformat()
        created_date = datetime.strptime(response['datecreated'], '%Y-%m-%d %H:%M:%S.%f').date().isoformat()
        self.assertEqual(created_date, current_date)
        self.client.transformDelete(response["uid"])
        self.client.userDelete(user_create_response["uid"])
    
    #@unittest.skip
    def test_get_transformer(self):
        user_create_response = self.client.userCreate("Test User", 100)
        trans_create_response = self.client.transformCreate("Test Transformer", {"chain":"_dbg_text2text"},owner=user_create_response['uid'])
        response = self.client.transformGet(trans_create_response['uid'])
        self.assertEqual(response["cfg"], {"chain":"_dbg_text2text"})
        self.assertEqual(response["name"], "Test Transformer")
        self.assertEqual(response["owner"], user_create_response['uid'])
        self.assertIsInstance(response["uid"], str)
        self.assertEqual(response['fees'], {'process_fees': {}, 'prompt_fees': {}})
        self.assertEqual(response["description"], None)
        self.assertEqual(response["mapio"], None)
        current_date = datetime.now().date().isoformat()
        created_date = datetime.strptime(response['datecreated'], '%Y-%m-%d %H:%M:%S.%f').date().isoformat()
        self.assertEqual(created_date, current_date)
        self.client.transformDelete(response["uid"])
        self.client.userDelete(user_create_response["uid"])
    
    #@unittest.skip
    def test_update_transformer(self):
        user_create_response = self.client.userCreate("Test User", 100)
        trans_create_response = self.client.transformCreate("Test Transformer", {"chain":"_dbg_text2text"},owner=user_create_response['uid'])
        trans_update_response = self.client.transformUpdate(trans_create_response["uid"],description="Test", fees={"prompt_fees":{user_create_response['uid']:5}})
        trans_get_response = self.client.transformGet(trans_create_response['uid'])
        self.assertEqual(trans_get_response["cfg"], {"chain":"_dbg_text2text"})
        self.assertEqual(trans_get_response["name"], "Test Transformer")
        self.assertEqual(trans_get_response["owner"], user_create_response['uid'])
        self.assertIsInstance(trans_get_response["uid"], str)
        self.assertEqual(trans_get_response['fees'], {"prompt_fees":{user_create_response['uid']:5}})
        self.assertEqual(trans_get_response["description"], "Test")
        self.assertEqual(trans_get_response["mapio"], None)
        current_date = datetime.now().date().isoformat()
        created_date = datetime.strptime(trans_get_response['datecreated'], '%Y-%m-%d %H:%M:%S.%f').date().isoformat()
        self.assertEqual(created_date, current_date)
        self.client.transformDelete(trans_get_response["uid"])
        self.client.userDelete(user_create_response["uid"])
    
    #@unittest.skip
    def test_delete_transformer(self):
        user_create_response = self.client.userCreate("Test User", 100)
        trans_create_response = self.client.transformCreate("Test Transformer", {"chain":"_dbg_text2text"},owner=user_create_response['uid'])
        response = self.client.transformDelete(trans_create_response["uid"])
        self.assertEqual(response, trans_create_response["uid"])
        #with self.assertRaises(json.JSONDecodeError):
        #    self.client.transformGet(trans_create_response['uid'])
        self.client.userDelete(user_create_response["uid"])
    
    #@unittest.skip
    def test_enum_transformer(self):
        user_create_response = self.client.userCreate("Test User", 100)
        trans_create_response = self.client.transformCreate("Test Transformer", {"chain":"_dbg_text2text"},owner=user_create_response['uid'])
        trans_enum_response = self.client.transformEnum(q=f"uid='{trans_create_response['uid']}'")
        self.assertEqual(trans_enum_response[0]["uid"], trans_create_response['uid'])
        self.client.transformDelete(trans_create_response["uid"])
        self.client.userDelete(user_create_response["uid"])
    
    #@unittest.skip
    def test_apply_transformer_doc(self):
        user_create_response = self.client.userCreate("Test User", 100)
        trans_create_response = self.client.transformCreate("Test Transformer", {"chain":"_dbg_text2text"},fees={"prompt_fees":{user_create_response['uid']:50}}, owner=user_create_response['uid'])
        doc_create_response = self.client.docCreate(content={"ssText":"test text"},owner=user_create_response['uid'])
        user2_create_response = self.client.userCreate("Test User 2", 100)
        trans_apply_response = self.client.transformApply(trans_create_response["uid"],user2_create_response['uid'], doc_create_response['uid'])
        self.assertEqual(self.client.userGet(user_create_response['uid'])["balance"], 150)
        self.assertEqual(self.client.userGet(user2_create_response['uid'])["balance"], 50)
        self.assertEqual(self.client.docFormat(trans_apply_response['did'])['content'], {"ssText":"test text[]"})
        self.client.docDelete(trans_apply_response["did"])
        self.client.docDelete(doc_create_response["uid"])
        self.client.transformDelete(trans_create_response["uid"])
        self.client.userDelete(user_create_response["uid"])
        self.client.userDelete(user2_create_response["uid"])
    
    #@unittest.skip
    def test_apply_transformer_content(self):
        user_create_response = self.client.userCreate("Test User", 100)
        trans_create_response = self.client.transformCreate("Test Transformer", {"chain":"_dbg_text2text"},fees={"prompt_fees":{user_create_response['uid']:50}}, owner=user_create_response['uid'])
        user2_create_response = self.client.userCreate("Test User 2", 100)
        trans_apply_response = self.client.transformApply(trans_create_response["uid"],user2_create_response['uid'], {"ssText":"test text"})
        self.assertEqual(self.client.userGet(user_create_response['uid'])["balance"], 150)
        self.assertEqual(self.client.userGet(user2_create_response['uid'])["balance"], 50)
        self.assertEqual(self.client.docFormat(trans_apply_response['did'])['content'], {"ssText":"test text[]"})
        self.client.docDelete(trans_apply_response["did"])
        self.client.transformDelete(trans_create_response["uid"])
        self.client.userDelete(user_create_response["uid"])
        self.client.userDelete(user2_create_response["uid"])
    
    #@unittest.skip
    def test_atomic_dbg_src(self):
        user_create_response = self.client.userCreate("Test User", 100)
        trans_create_response = self.client.transformCreate("Test Transformer", {"chain":"_dbg_src"},fees={"prompt_fees":{user_create_response['uid']:50}}, owner=user_create_response['uid'])
        user2_create_response = self.client.userCreate("Test User 2", 100)
        trans_apply_response = self.client.transformApply(trans_create_response["uid"],user2_create_response['uid'],{"ssText":""})
        self.assertEqual(self.client.userGet(user_create_response['uid'])["balance"], 150)
        self.assertEqual(self.client.userGet(user2_create_response['uid'])["balance"], 50)
        self.assertEqual(self.client.docFormat(trans_apply_response['did'])['content'], {"ssText":"text output"})
        self.client.docDelete(trans_apply_response["did"])
        self.client.transformDelete(trans_create_response["uid"])
        self.client.userDelete(user_create_response["uid"])
        self.client.userDelete(user2_create_response["uid"])
    
    #@unittest.skip
    def test_atomic_dbg_text2text(self):
        user_create_response = self.client.userCreate("Test User", 100)
        trans_create_response = self.client.transformCreate("Test Transformer", {"chain":"_dbg_text2text"},fees={"prompt_fees":{user_create_response['uid']:50}}, owner=user_create_response['uid'])
        user2_create_response = self.client.userCreate("Test User 2", 100)
        trans_apply_response = self.client.transformApply(trans_create_response["uid"],user2_create_response['uid'], {"ssText":"test text"})
        self.assertEqual(self.client.userGet(user_create_response['uid'])["balance"], 150)
        self.assertEqual(self.client.userGet(user2_create_response['uid'])["balance"], 50)
        self.assertEqual(self.client.docFormat(trans_apply_response['did'])['content'], {"ssText":"test text[]"})
        self.client.docDelete(trans_apply_response["did"])
        self.client.transformDelete(trans_create_response["uid"])
        self.client.userDelete(user_create_response["uid"])
        self.client.userDelete(user2_create_response["uid"])
    
    #@unittest.skip
    def test_atomic_dbg_text2image(self):
        user_create_response = self.client.userCreate("Test User", 100)
        trans_create_response = self.client.transformCreate("Test Transformer", {"chain":"_dbg_text2image"},fees={"prompt_fees":{user_create_response['uid']:50}}, owner=user_create_response['uid'])
        user2_create_response = self.client.userCreate("Test User 2", 100)
        trans_apply_response = self.client.transformApply(trans_create_response["uid"],user2_create_response['uid'], {"ssText":"test text"})
        self.assertEqual(self.client.userGet(user_create_response['uid'])["balance"], 150)
        self.assertEqual(self.client.userGet(user2_create_response['uid'])["balance"], 50)
        test_image_path="image2text.jpg"
        doc_format_response = self.client.docFormat(trans_apply_response['did'],fmt="json_base64")
        self.assertEqual(doc_format_response["content"]["bjImage"], file2b64(test_image_path))
        self.client.docDelete(trans_apply_response["did"])
        self.client.transformDelete(trans_create_response["uid"])
        self.client.userDelete(user_create_response["uid"])
        self.client.userDelete(user2_create_response["uid"])
    
    #@unittest.skip
    def test_atomic_dbg_img2text(self):
        user_create_response = self.client.userCreate("Test User", 100)
        trans_create_response = self.client.transformCreate("Test Transformer", {"chain":"_dbg_img2text"},fees={"prompt_fees":{user_create_response['uid']:50}}, owner=user_create_response['uid'])
        user2_create_response = self.client.userCreate("Test User 2", 100)
        test_image_path="image2text.jpg"
        trans_apply_response = self.client.transformApply(trans_create_response["uid"],user2_create_response['uid'], {"bjImage":file2b64(test_image_path)})
        self.assertEqual(self.client.userGet(user_create_response['uid'])["balance"], 150)
        self.assertEqual(self.client.userGet(user2_create_response['uid'])["balance"], 50)
        doc_format_response = self.client.docFormat(trans_apply_response['did'])
        self.assertEqual(doc_format_response["content"]["ssText"], f"len of in image is {len(file2b64(test_image_path))}")
        self.client.docDelete(trans_apply_response["did"])
        self.client.transformDelete(trans_create_response["uid"])
        self.client.userDelete(user_create_response["uid"])
        self.client.userDelete(user2_create_response["uid"])
    
    #@unittest.skip
    def test_atomic_openai_text2text(self):
        user_create_response = self.client.userCreate("Test User", 100)
        trans_create_response = self.client.transformCreate("Test Transformer", {"chain":"_openai_text2text"},fees={"prompt_fees":{user_create_response['uid']:50}}, owner=user_create_response['uid'])
        user2_create_response = self.client.userCreate("Test User 2", 100)
        trans_apply_response = self.client.transformApply(trans_create_response["uid"],user2_create_response['uid'], {"ssText":"generate joke about football"})
        self.client.docPollForStatus(trans_apply_response["did"],max_duration_ms=20000,poll_interval_ms=5000, status="ready")
        self.assertEqual(self.client.userGet(user_create_response['uid'])["balance"], 150)
        self.assertEqual(self.client.userGet(user2_create_response['uid'])["balance"], 50)
        self.assertEqual(self.client.docFormat(trans_apply_response['did'])['content']['prompt'], "generate joke about football")
        self.client.docDelete(trans_apply_response["did"])
        self.client.transformDelete(trans_create_response["uid"])
        self.client.userDelete(user_create_response["uid"])
        self.client.userDelete(user2_create_response["uid"])
    
    #@unittest.skip
    def test_atomic_openai_text2image(self):
        user_create_response = self.client.userCreate("Test User", 100)
        trans_create_response = self.client.transformCreate("Test Transformer", {"vars": {"size":"256x256"},"chain":"_openai_text2image"},fees={"prompt_fees":{user_create_response['uid']:50}}, owner=user_create_response['uid'])
        user2_create_response = self.client.userCreate("Test User 2", 100)
        trans_apply_response = self.client.transformApply(trans_create_response["uid"],user2_create_response['uid'], {"ssText":"generate cute dog"})
        self.client.docPollForStatus(trans_apply_response["did"],max_duration_ms=20000,poll_interval_ms=5000, status="ready")
        self.assertEqual(self.client.userGet(user_create_response['uid'])["balance"], 150)
        self.assertEqual(self.client.userGet(user2_create_response['uid'])["balance"], 50)
        self.assertEqual(self.client.docFormat(trans_apply_response['did'])['content']['ssText'], "generate cute dog")
        self.assertIsInstance(self.client.docFormat(trans_apply_response['did'])['content']['bjImage'], str)
        self.client.docDelete(trans_apply_response["did"])
        self.client.transformDelete(trans_create_response["uid"])
        self.client.userDelete(user_create_response["uid"])
        self.client.userDelete(user2_create_response["uid"])
    
    #@unittest.skip
    def test_chain_mapping(self):
        # T(T(dbg_text2test), T(openai_text2text))
        user1_create_response = self.client.userCreate("Test User", 100)
        trans1_create_response = self.client.transformCreate("Test Transformer", {"chain":"_dbg_text2text"},fees={"prompt_fees":{user1_create_response['uid']:10}}, owner=user1_create_response['uid'])
        user2_create_response = self.client.userCreate("Test User", 100)
        trans2_create_response = self.client.transformCreate("Test Transformer", {"chain":"_openai_text2text"},fees={"prompt_fees":{user2_create_response['uid']:5}}, owner=user2_create_response['uid'])
        user3_create_response = self.client.userCreate("Test User", 100)
        trans3_create_response = self.client.transformCreate("Test Transformer", {"chain":[trans1_create_response['uid'], trans2_create_response['uid']]}, mapio={'1':{"ssText":"based on {{chain[\"0\"][\"out_doc\"][\"ssText\"]}}"}}, fees={"prompt_fees":{user3_create_response['uid']:5}}, owner=user3_create_response['uid'])
        trans_apply_response = self.client.transformApply(trans3_create_response["uid"],user1_create_response['uid'], {"ssText":"generate joke about ai"})
        self.client.docPollForStatus(trans_apply_response["did"],max_duration_ms=20000,poll_interval_ms=5000,status="ready")
        self.assertEqual(self.client.userGet(user1_create_response['uid'])["balance"], 90)
        self.assertEqual(self.client.userGet(user2_create_response['uid'])["balance"], 105)
        self.assertEqual(self.client.userGet(user3_create_response['uid'])["balance"], 105)
        self.assertIn("generate joke about ai", self.client.docFormat(trans_apply_response['did'])['content']['prompt'])
        self.assertIsInstance(self.client.docFormat(trans_apply_response['did'])['content']['ssText'], str)
        self.client.docDelete(trans_apply_response["did"])
        self.client.transformDelete(trans3_create_response["uid"])
        self.client.transformDelete(trans2_create_response["uid"])
        self.client.transformDelete(trans1_create_response["uid"])
        self.client.userDelete(user1_create_response["uid"])
        self.client.userDelete(user2_create_response["uid"])
        self.client.userDelete(user3_create_response["uid"])
    
    #@unittest.skip
    def test_complex_chain(self):
        # T(T(T(T(dbg_text2image),T(dbg_image2text)), T(dbg_text2text)), T(dbg_text2image))
        user1_create_response=self.client.userCreate(name="Test User",balance=1000)
        trans1_create_response=self.client.transformCreate("Test Transformer", {"chain":"_dbg_text2image"},owner=user1_create_response["uid"],fees={"prompt_fees":{user1_create_response["uid"]:15}})

        user2_create_response=self.client.userCreate(name="Test User",balance=1000)
        trans2_create_response=self.client.transformCreate("Test Transformer", {"chain":"_dbg_img2text"},owner=user2_create_response["uid"],fees={"prompt_fees":{user2_create_response["uid"]:5}})
    
        user3_create_response=self.client.userCreate(name="Test User",balance=1000)
        trans3_create_response=self.client.transformCreate("Test Transformer", {"chain":"_dbg_text2text"},owner=user3_create_response["uid"],fees={"prompt_fees":{user3_create_response["uid"]:2}})
    
        user4_create_response=self.client.userCreate(name="Test User",balance=1000)
        trans4_create_response=self.client.transformCreate("Test Transformer", {"chain":[trans1_create_response["uid"], trans2_create_response["uid"]]},owner=user4_create_response["uid"],fees={"prompt_fees":{user4_create_response["uid"]:4}})
        
        user5_create_response=self.client.userCreate(name="Test User",balance=1000)
        trans5_create_response=self.client.transformCreate("Test Transformer", {"chain":[trans4_create_response["uid"], trans3_create_response["uid"]]},owner=user5_create_response["uid"],fees={"prompt_fees":{user5_create_response["uid"]:1}})
        
        trans6_create_response=self.client.transformCreate("Test Transformer", {"chain":[trans5_create_response["uid"], trans1_create_response["uid"]]},owner=user2_create_response["uid"],fees={"prompt_fees":{user2_create_response["uid"]:30}})

        user6_create_response=self.client.userCreate(name="Test User",balance=1000)
        
        trans_apply_response = self.client.transformApply(trans6_create_response["uid"],user6_create_response['uid'], {"ssText":"sample text"})
        self.client.docPollForStatus(trans_apply_response["did"],max_duration_ms=20000,status="ready")
        self.assertEqual(self.client.userGet(user6_create_response['uid'])["balance"], 1000-30-1-4-2-5-15-15)
        self.assertEqual(self.client.userGet(user2_create_response['uid'])["balance"], 1000+30+5)
        self.assertEqual(self.client.userGet(user1_create_response['uid'])["balance"], 1000+15+15)
        self.assertEqual(self.client.userGet(user5_create_response['uid'])["balance"], 1000+1)
        self.assertEqual(self.client.userGet(user3_create_response['uid'])["balance"], 1000+2)
        self.assertEqual(self.client.userGet(user4_create_response['uid'])["balance"], 1000+4)
        
        test_image_path="image2text.jpg"
        doc_format_response = self.client.docFormat(trans_apply_response['did'],fmt="json_base64")
        self.assertEqual(doc_format_response["content"]["bjImage"], file2b64(test_image_path))
        
        self.client.docDelete(trans_apply_response["did"])
        self.client.transformDelete(trans6_create_response["uid"])
        self.client.transformDelete(trans5_create_response["uid"])
        self.client.transformDelete(trans4_create_response["uid"])
        self.client.transformDelete(trans3_create_response["uid"])
        self.client.transformDelete(trans2_create_response["uid"])
        self.client.transformDelete(trans1_create_response["uid"])
        self.client.userDelete(user1_create_response["uid"])
        self.client.userDelete(user2_create_response["uid"])
        self.client.userDelete(user3_create_response["uid"])
        self.client.userDelete(user4_create_response["uid"])
        self.client.userDelete(user5_create_response["uid"])
        self.client.userDelete(user6_create_response["uid"])
        
    ### FEEDS
    def test_create_feed(self):
        user_create_response = self.client.userCreate("Test User", 100)
        trans_create_response=self.client.transformCreate("Test Transformer", {"chain":"_dbg_text2image"},owner=user1_create_response["uid"])
        feed_create_response=self.client.feedAttach(name="Test Feed", description="Test Feed", owner=user_create_response['uid'], transformer=trans_create_response['uid'])
        self.assertEqual(feed_create_response["owner"], user_create_response['uid'])
        self.assertEqual(feed_create_response["transformer"], trans_create_response['uid'])
        self.assertEqual(feed_create_response["name"], 'Test Feed')
        self.assertIsInstance(feed_create_response["uid"], str)
        self.assertEqual(feed_create_response['fees'], {'process_fees': {}, 'prompt_fees': {}})
        current_date = datetime.now().date().isoformat()
        created_date = datetime.strptime(feed_create_response['datecreated'], '%Y-%m-%d %H:%M:%S.%f').date().isoformat()
        self.assertEqual(created_date, current_date)
        self.client.feedDelete(feed_create_response["uid"])
        self.client.transformDelete(trans_create_response["uid"])
        self.client.userDelete(user_create_response["uid"])
    
    
    def test_create_feed_trans_dict(self):
        user_create_response = self.client.userCreate("Test User", 100)
        feed_create_response=self.client.feedAttach(name="Test Feed", description="Test Feed", owner=user_create_response['uid'], transformer={"chain":"_dbg_text2image"})
        self.assertEqual(feed_create_response["owner"], user_create_response['uid'])
        self.assertEqual(feed_create_response["name"], 'Test Feed')
        self.assertIsInstance(feed_create_response["uid"], str)
        self.assertEqual(feed_create_response['fees'], {'process_fees': {}, 'prompt_fees': {}})
        current_date = datetime.now().date().isoformat()
        created_date = datetime.strptime(feed_create_response['datecreated'], '%Y-%m-%d %H:%M:%S.%f').date().isoformat()
        self.assertEqual(created_date, current_date)
        self.client.feedDelete(feed_create_response["uid"])
        self.client.transformDelete(feed_create_response["transformer"])
        self.client.userDelete(user_create_response["uid"])
    
    
    def test_update_feed(self):
        user_create_response = self.client.userCreate("Test User", 100)
        trans_create_response=self.client.transformCreate("Test Transformer", {"chain":"_dbg_text2image"},owner=user1_create_response["uid"])
        feed_create_response=self.client.feedAttach(name="Test Feed", description="Test Feed", owner=user_create_response['uid'], transformer=trans_create_response['uid'])
        feed_update_response=self.client.feedUpdate(feed_create_response['uid'],description="Test Updated Feed")
        self.assertEqual(feed_update_response["owner"], user_create_response['uid'])
        self.assertEqual(feed_update_response["transformer"], trans_create_response['uid'])
        self.assertEqual(feed_update_response["name"], 'Test Feed')
        self.assertEqual(feed_update_response["description"], 'Test Updated Feed')
        self.assertIsInstance(feed_update_response["uid"], str)
        self.assertEqual(feed_update_response['fees'], {'process_fees': {}, 'prompt_fees': {}})
        current_date = datetime.now().date().isoformat()
        created_date = datetime.strptime(feed_create_response['datecreated'], '%Y-%m-%d %H:%M:%S.%f').date().isoformat()
        self.assertEqual(created_date, current_date)
        self.client.feedDelete(feed_create_response["uid"])
        self.client.transformDelete(trans_create_response["uid"])
        self.client.userDelete(user_create_response["uid"])
    
    
    def test_update_feed(self):
        user_create_response = self.client.userCreate("Test User", 100)
        trans_create_response=self.client.transformCreate("Test Transformer", {"chain":"_dbg_text2image"},owner=user1_create_response["uid"])
        feed_create_response=self.client.feedAttach(name="Test Feed", description="Test Feed", owner=user_create_response['uid'], transformer=trans_create_response['uid'])
        feed_enum_response=self.client.feedEnum(q=f"uid='{feed_create_response['uid']}'")
        self.assertEqual(feed_enum_response[0]["uid"], feed_create_response['uid'])
        self.client.feedDelete(feed_create_response["uid"])
        self.client.transformDelete(trans_create_response["uid"])
        self.client.userDelete(user_create_response["uid"])
    
    
    def test_generate_feed(self):
        user1_create_response = self.client.userCreate("Test User", 100)
        trans_create_response = self.client.transformCreate("Test Transformer", {"chain":"_dbg_text2text", "in_doc":{"ssText": "test text"}}, fees={"prompt_fees":{user1_create_response['uid']:20}},owner=user1_create_response['uid'])
        user2_create_response = self.client.userCreate("Test User", 100)
        feed_create_response=self.client.feedAttach(name="Test Feed", description="Test Feed", owner=user2_create_response['uid'], transformer=trans_create_response['uid'])
        user3_create_response = self.client.userCreate("Test User", 100)
        
        feed_generate_response=self.client.feedGenerate(feed_create_response['uid'],uid=user3_create_response['uid'],maxnum=2)
        self.client.feedPollForStatus(feed_generate_response["taskid"],max_duration_ms=10000,poll_interval_ms=500, status="ready")
        feed_status_response=self.client.feedStatus(feed_create_response['uid'])
        self.assertEqual(feed_status_response["status"], 'ready')
        
        feed_user_docs_response = self.client.feedDocs(feed_create_response['uid'],user3_create_response['uid'])
        doc_format_response = self.client.docFormat(feed_user_docs_response[0]['uid'])
        self.assertEqual(len(feed_user_docs_response), 2)
        self.assertEqual(doc_format_response['content'], {"ssText":"test text[]"})
        
        feed_get_response = self.client.feedGet(feed_create_response['uid'])
        self.assertEqual(feed_get_response["transformer"], {"chain":"_dbg_text2text", "in_doc":{"ssText": "test text"}})
        current_date = datetime.now().date().isoformat()
        lastgen = datetime.strptime(feed_get_response['lastgen'], '%Y-%m-%d %H:%M:%S.%f').date().isoformat()
        self.assertEqual(lastgen, current_date)
        
        self.assertEqual(self.client.userGet(user1_create_response['uid'])["balance"], 140)
        self.assertEqual(self.client.userGet(user3_create_response['uid'])["balance"], 60)
        
        self.client.docDelete(feed_user_docs_response[0]['uid'])
        self.client.docDelete(feed_user_docs_response[1]['uid'])
        self.client.feedDelete(feed_create_response["uid"])
        self.client.transformDelete(trans_create_response["uid"])
        self.client.userDelete(user1_create_response["uid"])
        self.client.userDelete(user2_create_response["uid"])
        self.client.userDelete(user3_create_response["uid"])
    
    @unittest.skip
    def test_generate_feed_buy_rights(self):
        user1_create_response = self.client.userCreate("Test User", 100)
        trans_create_response = self.client.transformCreate("Test Transformer", {"chain":"_dbg_text2text", "in_doc":{"ssText": "test text"}}, fees={"prompt_fees":{user1_create_response['uid']:20}},owner=user1_create_response['uid'])
        user2_create_response = self.client.userCreate("Test User", 100)
        feed_create_response=self.client.feedAttach(name="Test Feed", description="Test Feed", owner=user2_create_response['uid'], transformer=trans_create_response['uid'])
        user3_create_response = self.client.userCreate("Test User", 100)
        
        feed_generate_response=self.client.feedGenerate(feed_create_response['uid'],uid=user3_create_response['uid'],maxnum=2)
        self.client.feedPollForStatus(feed_generate_response["taskid"],max_duration_ms=10000,poll_interval_ms=500, status="ready")        
        feed_user3_docs_response = self.client.feedDocs(feed_create_response['uid'],user3_create_response['uid'])
        
        user4_create_response = self.client.userCreate("Test User", 100)
        feed_generate2_response=self.client.feedGenerate(feed_create_response['uid'],uid=user4_create_response['uid'],maxnum=1)
        self.client.feedPollForStatus(feed_generate2_response["taskid"],max_duration_ms=10000,poll_interval_ms=500, status="ready")
        feed_user4_docs1_response = self.client.feedDocs(feed_create_response['uid'],user4_create_response['uid'])
        self.assertEqual(len(self.client.feedDocs(feed_create_response['uid'],user4_create_response['uid'])), 1)
        feed_generate2_response=self.client.feedGenerate(feed_create_response['uid'],uid=user4_create_response['uid'],maxnum=1)
        self.client.feedPollForStatus(feed_generate2_response["taskid"],max_duration_ms=10000,poll_interval_ms=500, status="ready")
        feed_user4_docs_response = self.client.feedDocs(feed_create_response['uid'],user4_create_response['uid'])
        self.assertEqual(len(feed_user4_docs_response), 1)
        self.assertEqual(feed_user4_docs_response[0], feed_user4_docs1_response[0])
        self.assertIn(feed_user4_docs_response[0], feed_user3_docs_response)
        
        user5_create_response = self.client.userCreate("Test User", 100)
        feed_generate3_response=self.client.feedGenerate(feed_create_response['uid'],uid=user5_create_response['uid'],maxnum=3)
        self.client.feedPollForStatus(feed_generate3_response["taskid"],max_duration_ms=10000,poll_interval_ms=500, status="ready")
        feed_user5_docs_response = self.client.feedDocs(feed_create_response['uid'],user5_create_response['uid'])
        self.assertEqual(len(feed_user5_docs_response), 3)
        self.assertIn(feed_user3_docs_response[0], feed_user5_docs_response)
        self.assertIn(feed_user3_docs_response[1], feed_user5_docs_response)
        
        doc_format_response = self.client.docFormat(feed_user4_docs_response[0]['uid'])
        self.assertEqual(doc_format_response['content'], {"ssText":"test text[]"})
        
        self.assertEqual(self.client.userGet(user1_create_response['uid'])["balance"], 220)
        self.assertEqual(self.client.userGet(user3_create_response['uid'])["balance"], 60)
        self.assertEqual(self.client.userGet(user4_create_response['uid'])["balance"], 80)
        self.assertEqual(self.client.userGet(user5_create_response['uid'])["balance"], 40)
    
    @unittest.skip # doesn't work properly
    def test_generate_feed_having_parent(self):
        user1_create_response = self.client.userCreate("Test User", 100)
        trans_create_response = self.client.transformCreate("Test Transformer", {"chain":"_dbg_text2text", "in_doc":{"ssText": "test text"}}, fees={"prompt_fees":{user1_create_response['uid']:20}},owner=user1_create_response['uid'])
        user2_create_response = self.client.userCreate("Test User", 100)
        feed_create_response=self.client.feedAttach(name="Test Feed", description="Test Feed", owner=user2_create_response['uid'], transformer=trans_create_response['uid'])
        user3_create_response = self.client.userCreate("Test User", 100)
        
        feed_generate_response=self.client.feedGenerate(feed_create_response['uid'],uid=user3_create_response['uid'],maxnum=2)
        self.client.feedPollForStatus(feed_generate_response["taskid"],max_duration_ms=10000,poll_interval_ms=500, status="ready")        
        feed_user3_docs_response = self.client.feedDocs(feed_create_response['uid'],user3_create_response['uid'])
        
        user4_create_response = self.client.userCreate("Test User", 100)
        trans2_create_response = self.client.transformCreate("Test Transformer", {"chain":"_dbg_text2text"}, fees={"prompt_fees":{user4_create_response['uid']:5}},owner=user4_create_response['uid'])
        user5_create_response = self.client.userCreate("Test User", 100)
        feed2_create_response=self.client.feedAttach(name="Test Feed", description="Test Feed", owner=user5_create_response['uid'], transformer=trans2_create_response['uid'], source=feed_create_response['uid'])
        
        user6_create_response = self.client.userCreate("Test User", 100)
        feed_generate2_response=self.client.feedGenerate(feed2_create_response['uid'],uid=user6_create_response['uid'],maxnum=1)
        self.client.feedPollForStatus(feed_generate2_response["taskid"],max_duration_ms=10000,poll_interval_ms=500, status="ready")
        feed_user6_docs_response = self.client.feedDocs(feed2_create_response['uid'],user6_create_response['uid'])
        self.assertEqual(len(feed_user6_docs_response), 1)
        doc_format_response = self.client.docFormat(feed_user6_docs_response[0]['uid'])
        self.assertEqual(doc_format_response['content'], {"ssText":"test text[][]"})
        
        self.assertEqual(self.client.userGet(user1_create_response['uid'])["balance"], 160)
        self.assertEqual(self.client.userGet(user3_create_response['uid'])["balance"], 60)
        self.assertEqual(self.client.userGet(user4_create_response['uid'])["balance"], 105)
        self.assertEqual(self.client.userGet(user6_create_response['uid'])["balance"], 75)
        